# Lección 13.1 — Discriminativo vs. generativo: el crítico y el artista

> **Módulo 13 · Modelos generativos y LLMs** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 5.2 (regresión logística), Lección 5.7 (Naive Bayes), Lección 8.1 (estructura sin etiquetas)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía la diferencia entre un modelo **discriminativo** y uno **generativo**.
- Entender qué significa **aprender una distribución** y **muestrear** de ella para crear datos nuevos.
- Construir un generador de texto mínimo (letra a letra) y relacionarlo con los LLMs.
- Reconocer el sobreajuste de un modelo generativo: **copiar** los datos de entrenamiento.
- Ubicar en un mapa las familias generativas del módulo: VAE, GAN, difusión y modelos autorregresivos.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KernelDensity, NearestNeighbors
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)   # semilla fija: tus números coincidirán con las respuestas
plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el crítico de arte y el pintor

Imagina dos expertos en Van Gogh:

- 🧐 **El crítico** mira un cuadro y dice *"es un Van Gogh"* o *"no lo es"*. Para eso le basta con fijarse en **lo que distingue** a Van Gogh de otros pintores (las pinceladas en remolino, ciertos amarillos). **No sabe pintar.**
- 🎨 **El pintor** ha estudiado tanto a Van Gogh que puede **pintar un cuadro nuevo** que parece suyo. Para eso necesita entender **cómo es** un Van Gogh por completo: composición, colores, trazos, temas.

En ML:
- Un modelo **discriminativo** es el crítico: aprende a **distinguir** (dado un dato, ¿qué etiqueta tiene?). Casi todo lo que viste hasta ahora (regresión logística, árboles, SVM, CNN) es discriminativo.
- Un modelo **generativo** es el pintor: aprende **cómo son los datos** y puede **crear ejemplos nuevos** que se parecen a ellos. ChatGPT, Stable Diffusion o un generador de voces son generativos.

### ✋ Ejercicio 1 — ¿Crítico o pintor?
Clasifica cada sistema como **discriminativo (D)** o **generativo (G)**:
1. Un filtro de spam.
2. Una app que crea fotos de caras de personas que no existen.
3. El autocompletar del celular.
4. Un modelo que estima el precio de una casa.
5. Un sistema que escribe la descripción de un producto a partir de sus características.

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuesta</b></summary>

1. **D**: solo decide "spam / no spam".
2. **G**: crea imágenes nuevas.
3. **G**: produce la siguiente palabra (y, encadenándolas, frases enteras). Es la misma idea de un LLM, en pequeño.
4. **D**: predice un número a partir de los datos de la casa; no crea casas.
5. **G**: crea texto nuevo.

Truco: si la **salida** es una etiqueta o un número, suele ser discriminativo; si la salida es **un dato completo** (una imagen, un texto, un audio), es generativo.
</details>

---
## 2. 📖 El concepto: frontera vs. distribución

Una frutería quiere distinguir 🍏 **manzanas** de 🍊 **naranjas** por su **peso** y su **diámetro**. Tenemos 100 frutas de cada tipo.

In [ ]:
n = 100
manzanas = rng.multivariate_normal([150, 7.5], [[300, 6], [6, 0.25]], n)   # peso (g), diámetro (cm)
naranjas = rng.multivariate_normal([200, 8.5], [[400, 7], [7, 0.30]], n)
X = np.vstack([manzanas, naranjas])
y = np.array([0] * n + [1] * n)          # 0 = manzana, 1 = naranja

plt.scatter(*manzanas.T, c="tab:green", label="🍏 manzanas", s=15)
plt.scatter(*naranjas.T, c="tab:orange", label="🍊 naranjas", s=15)
plt.xlabel("peso (g)"); plt.ylabel("diámetro (cm)"); plt.legend(); plt.show()

### 📖 Dos formas de aprender con los mismos datos
- **🧐 Discriminativo (regresión logística, Lección 5.2):** busca la **frontera** que separa manzanas de naranjas. Aprende directamente *"dado este peso y este diámetro, ¿qué probabilidad hay de que sea naranja?"*. En notación: **P(fruta | medidas)**.
- **🎨 Generativo:** aprende **cómo es una manzana típica** (su peso medio, su diámetro medio, cuánto varían y cómo se relacionan) y lo mismo para las naranjas. En notación: **P(medidas | fruta)**. Es como tener un "molde" de cada fruta.

Con el molde puedes hacer **dos cosas**: clasificar (¿a qué molde se parece más esta fruta?), igual que Naive Bayes en la Lección 5.7… y **fabricar frutas nuevas**.

In [ ]:
# 🧐 Discriminativo: solo aprende la frontera
critico = LogisticRegression().fit(X, y)

# 🎨 Generativo: un "molde" (una campana de Gauss en 2D) por cada fruta
molde = {fruta: GaussianMixture(n_components=1, random_state=0).fit(X[y == fruta]) for fruta in [0, 1]}

print(f"Exactitud del crítico (logística)      : {critico.score(X, y):.0%}")
# Para clasificar con los moldes: ¿qué molde hace más probable esta fruta?
pred_gen = np.argmax([molde[f].score_samples(X) for f in [0, 1]], axis=0)
print(f"Exactitud del pintor (comparando moldes): {(pred_gen == y).mean():.0%}")

### 🔮 Ejercicio 2 — ¿Quién puede inventar una manzana?
Los dos modelos clasifican casi igual de bien. Ahora le pediremos a cada uno **5 manzanas nuevas**. Antes de ejecutar: ¿cuál de los dos podrá hacerlo?

✍️ *Mi predicción:*

In [ ]:
nuevas_manzanas, _ = molde[0].sample(5)
print("🎨 Manzanas inventadas por el modelo generativo (peso g, diámetro cm):")
print(np.round(nuevas_manzanas, 1))

print("\n🧐 ¿Y la regresión logística? Solo tiene esto:", [m for m in dir(critico) if m.startswith("pred")])

<details><summary><b>👀 Ver respuesta</b></summary>

Solo el **generativo**. La regresión logística no tiene ningún método para crear datos: solo sabe `predict` (decidir) y `predict_proba` (cuánta confianza tiene). Conoce la frontera, pero no sabe **cómo es** una manzana.

El generativo produce manzanas de ≈120–155 g y ≈7–8 cm: creíbles, porque salen del "molde" que aprendió. A esta acción de sacar ejemplos nuevos de un modelo se le llama **muestrear** (*sampling*).
</details>

### ✋ Ejercicio 3 — El precio de pintar
El crítico solo necesita la frontera: unos pocos números. El pintor necesita describir **todo** el dato. Piensa en fotos de 1 megapíxel:
1. Para decir *"¿hay un gato?"*, ¿cuántos números debe producir el modelo?
2. Para **crear** la foto de un gato, ¿cuántos?
3. ¿Cuál de las dos tareas crees que es más difícil y necesita más datos y cómputo?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Uno** (la probabilidad de "gato").
2. **Unos 3 millones** (1 millón de píxeles × 3 colores), y además coherentes entre sí: los bigotes tienen que estar donde va la cara.
3. **Generar**, con diferencia. Por eso los modelos generativos de imágenes y texto son enormes y se entrenan con cantidades gigantescas de datos. El crítico puede ignorar todo lo que no ayude a distinguir; el pintor no puede ignorar nada.
</details>

---
## 3. 📖 Cómo funciona: aprender una distribución y muestrear

"Aprender cómo son los datos" tiene un nombre técnico: **aprender la distribución** de los datos, es decir, qué valores son frecuentes y cuáles raros. Y **generar** = **muestrear** de esa distribución: tirar un dado cargado según lo aprendido.

Lo más sencillo es hacerlo con **texto, letra a letra**. Este es el mismo principio de los LLMs (Lección 13.5), pero a escala de juguete:
1. Contamos, en una lista de nombres, **qué letra suele venir después de cada letra**.
2. Para generar: empezamos con un "inicio", tiramos el dado para elegir la siguiente letra según esas frecuencias, y repetimos hasta que salga "fin".

A esto se le llama un **modelo de bigramas** (pares de letras).

In [ ]:
nombres = """sofia valentina isabella camila valeria mariana gabriela daniela victoria martina
luciana ximena maria jimena natalia andrea fernanda paula renata regina emilia catalina
santiago mateo sebastian leonardo matias emiliano diego daniel miguel alejandro nicolas
samuel david lucas benjamin tomas joaquin gabriel adrian martin julian rodrigo andres
pablo felipe ignacio manuel carlos javier fernando ricardo eduardo hugo bruno lorenzo
elena lucia carmen rosa julia clara ines alba noelia adriana carolina monica patricia
""".split()

# 1) Contar qué letra sigue a cada letra ("^" = inicio, "$" = fin)
conteos = defaultdict(Counter)
for nombre in nombres:
    letras = "^" + nombre + "$"
    for actual, siguiente in zip(letras, letras[1:]):
        conteos[actual][siguiente] += 1

print(f"{len(nombres)} nombres de entrenamiento")
print("Después de 'q' viene:", dict(conteos["q"]))
print("Después de 'a' viene:", dict(conteos["a"].most_common(6)), "...")

### 🔮 Ejercicio 4 — Antes de generar
1. Según los conteos, después de una "q", ¿qué letra elegirá **siempre** el modelo?
2. Después de una "a", ¿qué es lo más probable: otra letra o terminar el nombre (`$`)?

✍️ *Mis predicciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **"u"**, siempre: en los datos solo hay una "q" (jo**qu**in) y va seguida de "u", así que el modelo lo toma como regla sin que nadie se lo diga. Con un solo ejemplo, eso es más suerte que sabiduría: los modelos generativos necesitan **muchos** datos.
2. **Terminar** (`$`, 33 veces) es la opción más frecuente tras una "a": muchos nombres de la lista acaban en "a". Pero no es la única: le siguen "n" (16), "r" (11), "l", "t"…
</details>

▶️ Ahora sí, **generamos** nombres nuevos tirando el dado letra a letra:

In [ ]:
def generar_nombre(rng):
    letra, nombre = "^", ""
    while True:
        opciones = conteos[letra]
        letras, frecuencias = list(opciones), np.array(list(opciones.values()))
        letra = rng.choice(letras, p=frecuencias / frecuencias.sum())   # 🎲 dado cargado
        if letra == "$":
            return nombre
        nombre += letra

rng_nombres = np.random.default_rng(1)   # ✏️ cambia la semilla
inventados = [generar_nombre(rng_nombres) for _ in range(15)]
for nombre in inventados:
    print(f"{nombre:<14} {'(¡copiado de los datos!)' if nombre in nombres else ''}")

### ✋ Ejercicio 5
1. ¿Te parecen nombres "en español"? ¿Por qué el modelo produce cosas que **suenan** bien aunque no existan?
2. ¿Qué defectos ves? ¿Qué información le falta al modelo para hacerlo mejor?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Algunos **suenan** a nombres en español (*breo, belo, felblina, bril*), porque el modelo aprendió qué pares de letras son frecuentes ("el", "li", "na", terminaciones en "a" u "o"). Ninguno de los 15 está copiado de la lista: son **nuevos**.
2. Muchos son malos: demasiado cortos ("s", "ma") o larguísimos ("melicatanatiateja"), o empiezan raro ("rdro", "njiniano"): el modelo **solo mira la última letra**, así que no "recuerda" cómo empezó el nombre. La solución es **mirar más contexto**: dos, tres, diez letras atrás… o, como hace un LLM, miles de tokens atrás usando atención (Módulo 12). **Un LLM es, en esencia, este mismo juego llevado al extremo:** predecir el siguiente token con muchísimo contexto y miles de millones de parámetros.
</details>

---
## 4. 📖 ¿Qué pasa si… el pintor memoriza?

En los modelos discriminativos, sobreajustar significa memorizar las etiquetas de entrenamiento (Lección 4.4). En los generativos, sobreajustar significa algo más inquietante: **copiar** los datos de entrenamiento en lugar de crear datos nuevos. Eso es un problema de **privacidad** (puede "regurgitar" datos personales) y de **derechos de autor** (puede reproducir obras protegidas).

Lo veremos con un generador muy flexible, la **estimación de densidad por núcleos** (*Kernel Density Estimation*, KDE): coloca una pequeña "nube" alrededor de **cada** punto de entrenamiento y genera muestreando de esas nubes. Su perilla principal es el **ancho de banda** (`bandwidth`): el tamaño de cada nube.

Usaremos datos con forma de dos lunas.

In [ ]:
X_lunas, _ = make_moons(n_samples=300, noise=0.08, random_state=0)
X_tr, X_te = train_test_split(X_lunas, test_size=0.5, random_state=0)
plt.figure(figsize=(5, 3)); plt.scatter(*X_tr.T, s=8); plt.title("Datos de entrenamiento"); plt.show()

### 🔮 Ejercicio 6 — Nubes grandes y pequeñas
Generaremos 300 puntos con `bandwidth` = **0.01**, **0.1** y **0.6**.
1. ¿Con cuál crees que los puntos generados formarán una mancha que ya no parece dos lunas?
2. ¿Con cuál crees que los puntos generados serán casi **copias** de los de entrenamiento?

✍️ *Mis predicciones:*

In [ ]:
def distancia_a_copias(generados, reales):
    """Distancia media de cada punto generado al punto real más cercano."""
    vecino = NearestNeighbors(n_neighbors=1).fit(reales)
    return vecino.kneighbors(generados)[0].mean()

ANCHOS = [0.01, 0.1, 0.6]    # ✏️ prueba otros valores
fig, axes = plt.subplots(1, len(ANCHOS), figsize=(13, 3.5))
for ax, bw in zip(axes, ANCHOS):
    kde = KernelDensity(bandwidth=bw).fit(X_tr)
    generados = kde.sample(300, random_state=0)
    ax.scatter(*X_tr.T, s=8, c="lightgray", label="reales")
    ax.scatter(*generados.T, s=8, c="tab:red", label="generados")
    ax.set_title(f"bandwidth={bw}\ndist. al real más cercano: {distancia_a_copias(generados, X_tr):.3f}")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. **0.6**: nubes tan grandes que todo se mezcla en una mancha. Es **subajuste**: el modelo es demasiado simple para la forma de los datos.
2. **0.01**: nubes diminutas; cada punto generado cae prácticamente **encima** de un punto real (distancia media ≈0.01). No está creando: está **copiando**. Es el **sobreajuste** de un modelo generativo.

Con **0.1** se respeta la forma de las lunas y los puntos son nuevos (≈0.07 del real más cercano). Con 0.6 la distancia es ≈0.32: puntos lejos de cualquier dato real.
</details>

### 📖 ¿Cómo se evalúa un pintor?
Con un clasificador medías exactitud en datos de prueba. Con un generador **no hay respuesta correcta**: ¿qué nota le pones a un cuadro inventado? Algunas ideas:
- **Verosimilitud en datos nuevos** (*likelihood*): ¿qué tan probables considera el modelo los datos **de prueba** que nunca vio? Un buen molde debería encontrarlos "normales". Un modelo que solo copia los encuentra rarísimos.
- **Métricas de muestra** (en imágenes, por ejemplo *FID*): comparan estadísticamente las muestras generadas con datos reales.
- **Evaluación humana**: personas que puntúan o comparan resultados. Es lo que se usa con los LLMs (Lección 13.5).

Probemos la primera, que funciona igual que la validación de siempre.

### 🔮 Ejercicio 7 — El examen del pintor
Calcularemos la **log-verosimilitud media** en entrenamiento y en prueba para varios anchos (más alto = mejor). Antes de ejecutar: ¿qué ancho crees que tendrá la mejor nota **en prueba**? ¿Y en **entrenamiento**?

✍️ *Mis predicciones:*

In [ ]:
print(f"{'bandwidth':>10} {'entrenamiento':>14} {'prueba':>10}")
for bw in [0.01, 0.03, 0.05, 0.1, 0.2, 0.6]:     # ✏️
    kde = KernelDensity(bandwidth=bw).fit(X_tr)
    print(f"{bw:>10} {kde.score_samples(X_tr).mean():>14.2f} {kde.score_samples(X_te).mean():>10.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- En **entrenamiento**, gana el ancho más pequeño (0.01) con muchísima diferencia: cada punto tiene su propia nube encima.
- En **prueba**, el 0.01 es **el peor** (nota muy negativa): los puntos nuevos caen entre las nubecitas, donde el modelo cree que "no puede haber nada". La mejor nota en prueba está en **0.1** (≈ −1.05), con 0.05 cerca.

¡Es la misma curva en "U" de siempre (Lección 4.4)! Entrenamiento mejora sin fin, prueba mejora y luego empeora. **Los modelos generativos también sobreajustan, y se detecta igual: con datos que el modelo no vio.**
</details>

---
## 5. 🗺️ El mapa de la familia generativa

Generar manzanas en 2D o nombres letra a letra es fácil. Generar **fotos** o **párrafos** coherentes es muchísimo más difícil, y para eso existen familias de modelos basadas en redes neuronales. Las verás en este módulo:

| Familia | Idea en una frase | Analogía | Lección |
|---|---|---|---|
| **Autoencoder / VAE** | Comprimir el dato en un código pequeño y aprender a reconstruirlo; muestrear códigos nuevos | Un resumen que permite reescribir el libro | 13.2 |
| **GAN** | Un generador intenta engañar a un discriminador que aprende a detectar falsos | El falsificador y el detective | 13.3 |
| **Difusión** | Aprender a quitar ruido poco a poco, partiendo de ruido puro | Un escultor que quita lo que sobra | 13.4 |
| **Autorregresivo** (LLMs) | Generar una pieza (token) a la vez, cada una según las anteriores | El autocompletar, llevado al extremo | 13.5–13.9 |

> 💡 Fíjate en algo curioso: la **GAN** usa un **discriminativo** (el detective) como entrenador del generativo. Las dos ideas de esta lección trabajan juntas.

### ✋ Ejercicio 8 — Ubica a cada uno
¿A qué familia del mapa pertenece cada generador que construimos en esta lección?
1. El generador de nombres letra a letra.
2. Un modelo que crea una imagen empezando por "nieve" de televisión y limpiándola en 50 pasos.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Autorregresivo**: genera una pieza (letra) a la vez, cada una según la anterior. Es un "mini-LLM" de un solo carácter de contexto.
2. **Difusión** (Lección 13.4): parte de ruido puro y lo va limpiando paso a paso.
</details>

---
## 6. 🎛️ Perillas de los generadores de esta lección

Cada familia tendrá su propia tabla en su lección. Aquí, las de los dos generadores "clásicos" que usamos:

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `bandwidth` (KDE) | Tamaño de la nube alrededor de cada punto | Muestras más borrosas → **subajuste** (mancha) | Muestras pegadas a los datos → **copia** (sobreajuste) | Elegir con la verosimilitud en validación |
| `n_components` (mezcla de gaussianas) | Cuántos "moldes" se combinan | Más flexible → puede sobreajustar | Más simple → puede no capturar la forma | Elegir con verosimilitud en validación o BIC |
| Contexto (modelo de n-gramas) | Cuántos símbolos anteriores mira | Más coherente, pero necesita muchos más datos (y empieza a copiar) | Más "inventivo" e incoherente | 2–3 letras en un juguete; los LLMs usan miles de tokens |

### ✏️ Ejercicio 9 — Más moldes
Una sola campana de Gauss (un "molde") no puede dibujar dos lunas. Cambia `N_MOLDES` a 1, 4 y 16 y observa las muestras y la nota en prueba.

In [ ]:
N_MOLDES = 4    # ✏️ prueba 1, 4 y 16
gmm = GaussianMixture(n_components=N_MOLDES, random_state=0).fit(X_tr)
generados, _ = gmm.sample(300)
plt.figure(figsize=(5, 3))
plt.scatter(*X_tr.T, s=8, c="lightgray"); plt.scatter(*generados.T, s=8, c="tab:purple")
plt.title(f"{N_MOLDES} moldes · log-verosimilitud en prueba: {gmm.score(X_te):.2f}"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **1 molde**: una elipse que cubre las dos lunas **y el hueco entre ellas**: genera puntos donde nunca hubo datos. Nota en prueba ≈ −2.0 (la peor).
- **4 moldes**: ya se ven las lunas, con algunos puntos fuera de lugar. Nota ≈ −1.3 (**la mejor**).
- **16 moldes**: ¡las muestras se ven aún más bonitas!, pero la nota en prueba **baja** (≈ −1.5). Con solo 150 puntos, 16 moldes empiezan a ajustarse a los puntos concretos de entrenamiento: sobreajuste.

Moraleja: **a ojo** habrías elegido 16. La verosimilitud en datos de prueba te avisa de que ya estás memorizando. Igual que con los modelos discriminativos, se elige con validación.
</details>

---
## 7. 🚨 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Las muestras generadas son casi idénticas a ejemplos de entrenamiento | Sobreajuste del generador: **memoriza y copia** | Modelo menos flexible, más datos, comprobar distancia al ejemplo real más cercano |
| Las muestras son "promedios" borrosos, sin detalle | Modelo demasiado simple (subajuste) | Modelo más flexible o más capacidad |
| Muestras en zonas donde nunca hubo datos | El modelo no capta la forma (p. ej. una sola gaussiana para dos lunas) | Más componentes / otra familia de modelo |
| "Mi generador tiene 99 % de exactitud" | Usar métricas de clasificación para un generador | Verosimilitud en datos de prueba, métricas de muestra, evaluación humana |
| Usar un generativo para una tarea que solo necesita una etiqueta | Herramienta excesiva | Si solo necesitas decidir, un discriminativo suele ser más simple y preciso |

### 🔮 Ejercicio 10 — ¿Crea o copia?
Un equipo entrena un generador de nombres con contexto de **6 letras** en lugar de 1. ¿Qué esperas que pase con la proporción de nombres generados que ya estaban en la lista de entrenamiento?

✍️ *Mi predicción:*

In [ ]:
def entrenar_ngramas(nombres, contexto):
    tabla = defaultdict(Counter)
    for nombre in nombres:
        letras = "^" * contexto + nombre + "$"
        for i in range(len(letras) - contexto):
            tabla[letras[i:i + contexto]][letras[i + contexto]] += 1
    return tabla

def generar_con(tabla, contexto, rng):
    previo, nombre = "^" * contexto, ""
    while True:
        opciones = tabla[previo]
        letra = rng.choice(list(opciones), p=np.array(list(opciones.values())) / sum(opciones.values()))
        if letra == "$":
            return nombre
        nombre += letra
        previo = previo[1:] + letra

for contexto in [1, 2, 3, 6]:      # ✏️
    tabla = entrenar_ngramas(nombres, contexto)
    rng_c = np.random.default_rng(0)
    muestra = [generar_con(tabla, contexto, rng_c) for _ in range(200)]
    copiados = np.mean([m in nombres for m in muestra])
    print(f"contexto={contexto}: {copiados:5.0%} copiados · ejemplos: {', '.join(muestra[:6])}")

<details><summary><b>👀 Ver respuesta</b></summary>

La proporción de copias **sube** con el contexto: con 1 letra, **0 %** copiados (aunque los nombres son poco creíbles); con 2, ≈12 %; con 3, ≈67 % (*eduardo, ines, hugo…*); con 6, **el 100 %** son copias exactas de la lista. Con tan pocos datos (70 nombres), cada contexto largo solo aparece una vez, así que el modelo no tiene más opción que seguir el único camino que vio.

Esta es la tensión de todo modelo generativo: **más capacidad → resultados más realistas, pero también más riesgo de copiar.** Los LLMs reales lo compensan con cantidades enormes de datos, y aun así a veces reproducen fragmentos memorizados (Lección 15.5).
</details>

---
## 8. 📋 Ficha: discriminativo vs. generativo

| | 🧐 Discriminativo | 🎨 Generativo |
|---|---|---|
| **Aprende** | La frontera: P(etiqueta \| dato) | Cómo son los datos: P(dato), o P(dato \| etiqueta) |
| **Puede** | Clasificar, predecir números | Crear datos nuevos (y a veces también clasificar) |
| **Ejemplos** | Regresión logística, árboles, SVM, CNN clasificadora, BERT para clasificar | Naive Bayes, mezcla de gaussianas, VAE, GAN, difusión, GPT |
| **Dificultad** | Menor: puede ignorar lo que no ayuda a distinguir | Mayor: tiene que modelar **todo** el dato |
| **Cómo se evalúa** | Exactitud, F1, error… en datos de prueba | Verosimilitud en prueba, métricas de muestra, evaluación humana |
| **Sobreajuste se ve como** | Brecha entre entrenamiento y prueba | Copiar ejemplos de entrenamiento |
| **Úsalo cuando…** | Solo necesitas una decisión o un número | Necesitas crear contenido, datos sintéticos, completar datos faltantes o detectar lo "raro" |

## 9. 🧠 Resumen en 5 puntos
1. Un modelo **discriminativo** aprende a **distinguir** (la frontera); uno **generativo** aprende **cómo son los datos** y puede **crear** ejemplos nuevos.
2. Generar = **muestrear** de la distribución aprendida: tirar un dado cargado según lo que el modelo aprendió.
3. Un generador de texto **letra a letra** es un LLM en miniatura: predecir lo siguiente y repetir. Lo que cambia en un LLM es **cuánto contexto** mira y su tamaño.
4. Los generativos también **sobreajustan**, y eso se ve como **copiar** el entrenamiento: un riesgo de privacidad y de derechos de autor. Se detecta con datos de prueba y midiendo la distancia a los ejemplos reales.
5. Las grandes familias generativas modernas son **VAE**, **GAN**, **difusión** y **autorregresivos** (LLMs); las verás en el resto del módulo.

---
## 10. 🏋️ Ejercicios finales

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál de estos modelos es **generativo**?
- a) Un árbol de decisión que aprueba créditos · b) Una CNN que reconoce dígitos · c) Un modelo que crea música nueva · d) Una regresión lineal de precios

**2.** "Muestrear" de un modelo generativo significa…
- a) Elegir una muestra de los datos de entrenamiento · b) Sacar un ejemplo nuevo según las probabilidades que aprendió · c) Medir su exactitud · d) Dividir los datos en entrenamiento y prueba

**3.** Tu generador de caras produce, a veces, fotos **idénticas** a personas reales del conjunto de entrenamiento. Esto indica…
- a) Subajuste · b) Sobreajuste (memorización) · c) Que el modelo es discriminativo · d) Que funciona perfectamente

**4.** En el KDE, ¿qué valor de `bandwidth` produce una mancha que no respeta la forma de los datos?
- a) Muy pequeño · b) Muy grande · c) Cualquiera · d) Ninguno

**5.** ¿Qué idea comparten el generador de nombres de esta lección y GPT?
- a) Ambos usan atención · b) Ambos generan una pieza a la vez según las anteriores · c) Ambos son discriminativos · d) Ambos tienen miles de millones de parámetros

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** Crea datos nuevos (música). Los demás producen una etiqueta o un número.
2. **b)** Tirar el "dado cargado" que aprendió el modelo para producir un ejemplo nuevo.
3. **b)** El generador copia en lugar de crear: es la forma que toma el sobreajuste en los generativos.
4. **b)** Nubes enormes que se mezclan: subajuste.
5. **b)** Ambos son **autorregresivos**. El generador de nombres mira una letra atrás; GPT mira miles de tokens con atención.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Para detectar transacciones fraudulentas casi nunca tienes ejemplos de fraude, pero sí millones de transacciones normales. ¿Cómo podrías usar un modelo **generativo** de las transacciones normales para detectar fraudes? (Pista: piensa en la verosimilitud del Ejercicio 7 y en la Lección 8.6.)

**B. Caso de análisis 💼** Un hospital quiere compartir datos con una universidad, pero no puede entregar datos reales de pacientes. Alguien propone entrenar un modelo generativo y entregar **datos sintéticos**. ¿Qué ventaja tiene la idea? ¿Qué riesgo de esta lección deberían comprobar antes de entregar nada?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Entrenas un generativo solo con transacciones normales; aprende "cómo es una transacción normal". Para cada transacción nueva calculas su **verosimilitud**: si el modelo la considera muy improbable, es **rara** y merece revisión. Es la idea de la detección de anomalías (Lección 8.6), y la volverás a ver con autoencoders (Lección 13.2).

**B.** Ventaja: los investigadores reciben datos con las mismas **relaciones estadísticas** (edad, presión, diagnóstico…) sin ver a ningún paciente real. Riesgo: si el generador **memorizó**, algunos "pacientes sintéticos" pueden ser copias casi exactas de pacientes reales, y eso filtra datos privados. Antes de entregar hay que medir, por ejemplo, la **distancia de cada registro sintético al paciente real más cercano** (como en el Ejercicio 6) y comprobar que los datos sintéticos siguen siendo útiles (siguiente práctica).
</details>

### 10.3 🧩 Práctica integradora: "Datos sintéticos para entrenar un clasificador"

Una empresa tiene datos de clientes con dos características y una etiqueta (**compra / no compra**), pero no puede compartirlos con un proveedor externo. Quiere saber si unos **datos sintéticos** servirían para que el proveedor entrene un modelo. Lo harás en 5 pasos cortos. **No avances sin revisar cada respuesta.**

**Paso 1 ▶️ — Los datos reales**

In [ ]:
X_real, y_real = make_moons(n_samples=600, noise=0.25, random_state=3)
X_r_tr, X_r_te, y_r_tr, y_r_te = train_test_split(X_real, y_real, test_size=0.5, random_state=0)
plt.figure(figsize=(5, 3)); plt.scatter(*X_r_tr.T, c=y_r_tr, cmap="coolwarm", s=10)
plt.title("Clientes reales (rojo = compra)"); plt.show()
print(f"{len(X_r_tr)} clientes para entrenar, {len(X_r_te)} guardados para la prueba final")

**Paso 2 🔮 — La referencia.** Entrenamos un árbol de decisión con los datos **reales** y lo evaluamos en la prueba. Esta es la nota a la que aspiran los datos sintéticos. ¿Qué exactitud esperas, aproximadamente?

In [ ]:
from sklearn.tree import DecisionTreeClassifier
modelo_real = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_r_tr, y_r_tr)
print(f"Árbol entrenado con datos REALES → exactitud en prueba: {modelo_real.score(X_r_te, y_r_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Alrededor del **95 %**. Esa es la referencia: un buen conjunto sintético debería acercarse.
</details>

**Paso 3 ✏️ — Fabrica clientes sintéticos.** Entrenamos un generativo **por clase** (una mezcla de gaussianas para "compra" y otra para "no compra") y fabricamos 300 clientes nuevos. Prueba `N_MOLDES` = 1 y 8.

In [ ]:
N_MOLDES = 1    # ✏️ prueba 1 y 8

def fabricar(X, y, n_moldes, n_por_clase=150):
    Xs, ys = [], []
    for clase in [0, 1]:
        g = GaussianMixture(n_components=n_moldes, random_state=0).fit(X[y == clase])
        muestras, _ = g.sample(n_por_clase)
        Xs.append(muestras); ys.append(np.full(n_por_clase, clase))
    return np.vstack(Xs), np.concatenate(ys)

X_sint, y_sint = fabricar(X_r_tr, y_r_tr, N_MOLDES)
plt.figure(figsize=(5, 3)); plt.scatter(*X_sint.T, c=y_sint, cmap="coolwarm", s=10)
plt.title(f"Clientes SINTÉTICOS ({N_MOLDES} moldes por clase)"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Con **1 molde** por clase, cada clase es una elipse: se pierde la forma de luna y las elipses se solapan. Con **8 moldes**, los clientes sintéticos dibujan dos lunas muy parecidas a las reales.
</details>

**Paso 4 🔮 — ¿Sirven?** Entrenamos el mismo árbol **solo con datos sintéticos** y lo evaluamos con los clientes **reales** de prueba (esta técnica se llama *train on synthetic, test on real*). ¿Cuánto crees que pierde cada versión frente al ≈95 % de referencia?

In [ ]:
for n_moldes in [1, 8]:
    X_sint, y_sint = fabricar(X_r_tr, y_r_tr, n_moldes)
    m = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_sint, y_sint)
    print(f"{n_moldes} molde(s) por clase → exactitud en clientes REALES: {m.score(X_r_te, y_r_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con **1 molde**, la exactitud cae a ≈87 %: los datos sintéticos no capturan la forma real, así que el árbol aprende una frontera equivocada. Con **8 moldes** llega a ≈93 %, muy cerca del ≈95 % que se obtiene con datos reales. **Un buen generador conserva la información útil para la tarea.**
</details>

**Paso 5 ✋ — El control de privacidad.** Falta lo más importante: comprobar que no estamos **copiando** clientes. Compara la distancia de cada cliente sintético al cliente real más cercano con la distancia típica entre clientes reales.

In [ ]:
X_sint, y_sint = fabricar(X_r_tr, y_r_tr, 8)
d_sint = distancia_a_copias(X_sint, X_r_tr)
# entre reales: distancia de cada cliente de prueba al cliente de entrenamiento más cercano
d_real = distancia_a_copias(X_r_te, X_r_tr)
print(f"Sintético → real más cercano : {d_sint:.3f}")
print(f"Real nuevo → real más cercano: {d_real:.3f}")

✍️ ¿Hay señales de copia? ¿Qué le dirías a la empresa?

<details><summary><b>👀 Ver respuesta</b></summary>

Las dos distancias son **parecidas** (≈0.07–0.08): un cliente sintético está tan cerca de los reales como lo estaría un cliente **nuevo** de verdad. No hay señales de copia. Si la distancia de los sintéticos fuera muchísimo menor (p. ej. 0.001), el generador estaría regurgitando clientes reales.

Mensaje para la empresa: *"Los datos sintéticos permiten entrenar un modelo casi tan bueno como con los datos reales (≈93 % vs. ≈95 %) y no son copias de clientes concretos. Aun así, con datos sensibles conviene usar técnicas formales de privacidad (Lección 15.5)."*
</details>

---
## 📚 Para profundizar
- 📺 **Video:** Andrej Karpathy, *The spelled-out intro to language modeling: building makemore* (inglés): el generador de nombres de esta lección, llevado mucho más lejos.
- 📘 **Documentación:** scikit-learn, guía de usuario, secciones *Density Estimation* y *Gaussian mixture models*.
- 📄 **Clásico:** Ng y Jordan (2001), *On Discriminative vs. Generative classifiers* (compara regresión logística y Naive Bayes).

---
⬅️ **Anterior:** [12.6 Proyecto del módulo: El aprendiz de Cervantes ✒️](../../nivel-4-avanzado/m12-transformers/12.6-proyecto-del-modulo.md) · ➡️ **Siguiente:** [13.2 Autoencoders y VAEs: comprimir y reconstruir](13.2-autoencoders-y-vaes.ipynb)